# Suite CNT — The container layout

`Container` takes the settings of each container's grid, a mutation of the logical layout, then places each child the grid gains in its next cell and, before each it loses is removed, closes the gap it leaves, with no coordinates of any kind. Each step is one mutation, a transaction expression the bench journals, and the container layout's adjustments to it, in order, which the step asserts; the suite's last case replays the journal in a fresh store, and finds the same adjustments. mbse-basis [CAP-238: container schemas and grid insertion](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-238.md) and [CAP-280: the grid in the logical layout](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-280), by the rules of [the diagrams proposal](https://github.com/pitaman71/mbse-basis/blob/main/proposals/DIAGRAMS.md).

In [ ]:
from support import Bench, text
from mbse.Diagrams.Framework import Container, Logical

b = Bench()

<a id="CNT-01"></a>
## CNT-01 · The install, after Logical's, defines the container layout's schemas, each valid, and run again writes nothing

In [ ]:
NAMES = ["Container.HasCells", "Container.OfCell", "Container.Holds", "Container.Rules", "Container.Enforces",
         "Container.OfRule", "Logical.OfGrid", "Logical.OfShape", "Logical.OfAdjacency"]
schemas = [b.store.registered(name) for name in NAMES]
assert [schema.validate() for schema in schemas] == [[]] * len(NAMES)
b.store.transaction(Container.install)
assert [b.store.registered(name) for name in NAMES] == schemas
assert "cell" in b.store.registered("Logical.OfShape").adjacencies
assert {"cells", "rules"} <= set(b.store.registered("Logical.OfGrid").adjacencies)
assert "rule" in b.store.registered("Logical.OfAdjacency").adjacencies

<a id="CNT-02"></a>
## CNT-02 · A container's grid takes its settings before its children, each of which then takes the next cell

In [ ]:
assert b.child('', 'cpu') == '[{"kind":"grid","container":"overview","span":3,"grows":"rows","cell_rotation":0.0},{"kind":"place","shape":"cpu","order":0,"to":[0,0]},{"kind":"extents","rows":1,"columns":1}]'
assert b.layout('cpu', 2) == '[{"kind":"grid","container":"cpu","span":2,"grows":"rows","cell_rotation":0.0}]'
assert b.child('cpu', 'a') == '[{"kind":"place","shape":"a","order":0,"to":[0,0]},{"kind":"extents","rows":1,"columns":1}]'
assert b.child('cpu', 'b') == '[{"kind":"place","shape":"b","order":1,"to":[0,1]},{"kind":"extents","rows":1,"columns":2}]'
assert b.child('cpu', 'c') == '[{"kind":"place","shape":"c","order":2,"to":[1,0]},{"kind":"extents","rows":2,"columns":2}]'
assert b.cells("cpu") == '[[[0,0],[["a",0]],null,1,1],[[0,1],[["b",1]],null,1,1],[[1,0],[["c",2]],null,1,1]]'
assert text([Container.cell_of(b.shape("cpu.b")), Container.cell_of(b.shape("cpu"))]) == '[[0,1],[0,0]]'

<a id="CNT-03"></a>
## CNT-03 · A child added takes the next cell, and the grid's extents grow only where it widens or lengthens

In [ ]:
assert b.child('cpu', 'd') == '[{"kind":"place","shape":"d","order":3,"to":[1,1]}]'
assert b.child('cpu', 'e') == '[{"kind":"place","shape":"e","order":4,"to":[2,0]},{"kind":"extents","rows":3,"columns":2}]'
assert b.cells("cpu") == '[[[0,0],[["a",0]],null,1,1],[[0,1],[["b",1]],null,1,1],[[1,0],[["c",2]],null,1,1],[[1,1],[["d",3]],null,1,1],[[2,0],[["e",4]],null,1,1]]'
assert b.child('', 'one') == '[{"kind":"place","shape":"one","order":1,"to":[0,1]},{"kind":"extents","rows":1,"columns":2}]'
assert b.layout('one', 3) == '[{"kind":"grid","container":"one","span":3,"grows":"rows","cell_rotation":0.0}]'
assert b.child('one', 'x') == '[{"kind":"place","shape":"x","order":0,"to":[0,0]},{"kind":"extents","rows":1,"columns":1}]'

<a id="CNT-04"></a>
## CNT-04 · A grid that grows columns fills each column first

In [ ]:
assert b.child('', 'bus') == '[{"kind":"place","shape":"bus","order":2,"to":[0,2]},{"kind":"extents","rows":1,"columns":3}]'
assert b.layout('bus', 2, 'columns') == '[{"kind":"grid","container":"bus","span":2,"grows":"columns","cell_rotation":0.0}]'
assert b.child('bus', 'a') == '[{"kind":"place","shape":"a","order":0,"to":[0,0]},{"kind":"extents","rows":1,"columns":1}]'
assert b.child('bus', 'b') == '[{"kind":"place","shape":"b","order":1,"to":[1,0]},{"kind":"extents","rows":2,"columns":1}]'
assert b.child('bus', 'c') == '[{"kind":"place","shape":"c","order":2,"to":[0,1]},{"kind":"extents","rows":2,"columns":2}]'
assert b.child('bus', 'd') == '[{"kind":"place","shape":"d","order":3,"to":[1,1]}]'
assert b.child('bus', 'e') == '[{"kind":"place","shape":"e","order":4,"to":[0,2]},{"kind":"extents","rows":2,"columns":3}]'
assert b.cells("bus") == '[[[0,0],[["a",0]],null,1,1],[[0,1],[["c",2]],null,1,1],[[0,2],[["e",4]],null,1,1],[[1,0],[["b",1]],null,1,1],[[1,1],[["d",3]],null,1,1]]'

<a id="CNT-05"></a>
## CNT-05 · A child removed from its container leaves its cell, the children after it move back one each, and it goes to the diagram's top level, deleting nothing

In [ ]:
assert b.unchild('cpu', 'b') == '[{"kind":"leave","shape":"b","from":[0,1]},{"kind":"move","shape":"c","from":[1,0],"to":[0,1]},{"kind":"move","shape":"d","from":[1,1],"to":[1,0]},{"kind":"move","shape":"e","from":[2,0],"to":[1,1]},{"kind":"extents","rows":2,"columns":2},{"kind":"place","shape":"b","order":3,"to":[1,0]},{"kind":"extents","rows":2,"columns":3}]'
assert b.cells("cpu") == '[[[0,0],[["a",0]],null,1,1],[[0,1],[["c",2]],null,1,1],[[1,0],[["d",3]],null,1,1],[[1,1],[["e",4]],null,1,1]]'
assert b.unchild('cpu', 'e') == '[{"kind":"leave","shape":"e","from":[1,1]},{"kind":"place","shape":"e","order":4,"to":[1,1]}]'
assert b.unchild('bus', 'a', 'd') == '[{"kind":"leave","shape":"a","from":[0,0]},{"kind":"move","shape":"b","from":[1,0],"to":[0,0]},{"kind":"move","shape":"c","from":[0,1],"to":[1,0]},{"kind":"move","shape":"d","from":[1,1],"to":[0,1]},{"kind":"move","shape":"e","from":[0,2],"to":[1,1]},{"kind":"extents","rows":2,"columns":2},{"kind":"leave","shape":"d","from":[0,1]},{"kind":"move","shape":"e","from":[1,1],"to":[0,1]},{"kind":"place","shape":"a","order":5,"to":[1,2]},{"kind":"place","shape":"d","order":6,"to":[2,0]},{"kind":"extents","rows":3,"columns":3}]'
assert b.cells("bus") == '[[[0,0],[["b",1]],null,1,1],[[0,1],[["e",4]],null,1,1],[[1,0],[["c",2]],null,1,1]]'
assert b.unchild('one', 'x') == '[{"kind":"leave","shape":"x","from":[0,0]},{"kind":"extents","rows":0,"columns":0},{"kind":"place","shape":"x","order":7,"to":[2,1]}]'
assert b.cells("one") == '[]'
assert text(b.problems("cpu", "bus", "one")) == '[[],[],[]]'

<a id="CNT-06"></a>
## CNT-06 · A grid's settings outside their values are its diagram's; settings changed rearrange its cells, in order; a child added twice changes nothing; a shape no grid holds is refused a cell

In [ ]:
assert b.child('', 'diagonal') == '[{"kind":"place","shape":"diagonal","order":8,"to":[2,2]}]'
assert b.layout('diagonal', 2, 'diagonal') == '[{"kind":"grid","container":"diagonal","span":2,"grows":"rows","cell_rotation":0.0}]'
assert b.child('', 'empty') == '[{"kind":"place","shape":"empty","order":9,"to":[3,0]},{"kind":"extents","rows":4,"columns":3}]'
assert b.layout('empty', 0) == '[{"kind":"grid","container":"empty","span":3,"grows":"rows","cell_rotation":0.0}]'
assert b.child('cpu', 'a') == '[]'
with b.store.transaction():
    loose = b.store.Logical.OfShape(lambda s: s.owner(lambda a: a.insert(lambda e: e.name("loose").owner(b.overview))))
try:
    with b.store.transaction():
        Container.added(loose)
except ValueError as error:
    refused = str(error)
try:
    with b.store.transaction():
        Container.added(b.shape("cpu.a"))
except ValueError as error:
    twice = str(error)
assert text([refused, twice, Container.cell_of(loose), b.queue("cpu.a")]) == '["a shape no grid holds has no cell: its container, or its diagram, has no grid","a child is placed once: it is in a cell already",null,"[]"]'
b.logical(["kids.x"])
assert b.layout('kids', 2) == '[{"kind":"grid","container":"kids","span":2,"grows":"rows","cell_rotation":0.0}]'
assert b.turn('kids', 90.0) == '[{"kind":"grid","container":"kids","span":2,"grows":"rows","cell_rotation":90.0}]'
assert text([Container.cell_of(b.shape("kids")), Container.cell_of(b.shape("kids.x"))]) == '[[3,1],[0,0]]'
again = Bench()
assert text([again.child('', name) for name in "abcde"]) == '["[{\\"kind\\":\\"grid\\",\\"container\\":\\"overview\\",\\"span\\":3,\\"grows\\":\\"rows\\",\\"cell_rotation\\":0.0},{\\"kind\\":\\"place\\",\\"shape\\":\\"a\\",\\"order\\":0,\\"to\\":[0,0]},{\\"kind\\":\\"extents\\",\\"rows\\":1,\\"columns\\":1}]","[{\\"kind\\":\\"place\\",\\"shape\\":\\"b\\",\\"order\\":1,\\"to\\":[0,1]},{\\"kind\\":\\"extents\\",\\"rows\\":1,\\"columns\\":2}]","[{\\"kind\\":\\"place\\",\\"shape\\":\\"c\\",\\"order\\":2,\\"to\\":[0,2]},{\\"kind\\":\\"extents\\",\\"rows\\":1,\\"columns\\":3}]","[{\\"kind\\":\\"place\\",\\"shape\\":\\"d\\",\\"order\\":3,\\"to\\":[1,0]},{\\"kind\\":\\"extents\\",\\"rows\\":2,\\"columns\\":3}]","[{\\"kind\\":\\"place\\",\\"shape\\":\\"e\\",\\"order\\":4,\\"to\\":[1,1]}]"]'
assert again.layout('', 2, 'columns') == '[{"kind":"grid","container":"overview","span":2,"grows":"columns","cell_rotation":0.0},{"kind":"move","shape":"b","from":[0,1],"to":[1,0]},{"kind":"move","shape":"c","from":[0,2],"to":[0,1]},{"kind":"move","shape":"d","from":[1,0],"to":[1,1]},{"kind":"move","shape":"e","from":[1,1],"to":[0,2]}]'
assert again.cells('') == '[[[0,0],[["a",0]],null,1,1],[[0,1],[["c",2]],null,1,1],[[0,2],[["e",4]],null,1,1],[[1,0],[["b",1]],null,1,1],[[1,1],[["d",3]],null,1,1]]'
assert again.layout('', 2, 'columns', 90.0) == '[{"kind":"grid","container":"overview","span":2,"grows":"columns","cell_rotation":90.0}]'
assert text(again.replay()) == '[]'

<a id="CNT-07"></a>
## CNT-07 · A container without a grid is given one by its first child, as its diagram's defaults say, or by layout before it: the diagram's own, too; a change of the defaults rearranges each grid that sets none

In [ ]:
assert b.cells("") == '[[[0,0],[["cpu",0]],null,1,1],[[0,1],[["one",1]],null,1,1],[[0,2],[["bus",2]],null,1,1],[[1,0],[["b",3]],null,1,1],[[1,1],[["e",4]],null,1,1],[[1,2],[["a",5]],null,1,1],[[2,0],[["d",6]],null,1,1],[[2,1],[["x",7]],null,1,1],[[2,2],[["diagonal",8]],null,1,1],[[3,0],[["empty",9]],null,1,1],[[3,1],[["kids",10]],null,1,1]]'
assert b.child('', 'solo') == '[{"kind":"place","shape":"solo","order":11,"to":[3,2]}]'
assert b.child('solo', 'x') == '[{"kind":"grid","container":"solo","span":3,"grows":"rows","cell_rotation":0.0},{"kind":"place","shape":"x","order":0,"to":[0,0]},{"kind":"extents","rows":1,"columns":1}]'
assert b.cells("solo") == '[[[0,0],[["x",0]],null,1,1]]'
assert b.defaults(1, "columns") == '[{"kind":"grid","container":"overview","span":1,"grows":"columns","cell_rotation":0.0},{"kind":"move","shape":"b","from":[1,0],"to":[0,3]},{"kind":"move","shape":"e","from":[1,1],"to":[0,4]},{"kind":"move","shape":"a","from":[1,2],"to":[0,5]},{"kind":"move","shape":"d","from":[2,0],"to":[0,6]},{"kind":"move","shape":"x","from":[2,1],"to":[0,7]},{"kind":"move","shape":"diagonal","from":[2,2],"to":[0,8]},{"kind":"move","shape":"empty","from":[3,0],"to":[0,9]},{"kind":"move","shape":"kids","from":[3,1],"to":[0,10]},{"kind":"move","shape":"solo","from":[3,2],"to":[0,11]},{"kind":"extents","rows":1,"columns":12},{"kind":"grid","container":"diagonal","span":2,"grows":"columns","cell_rotation":0.0},{"kind":"grid","container":"empty","span":1,"grows":"rows","cell_rotation":0.0},{"kind":"grid","container":"solo","span":1,"grows":"columns","cell_rotation":0.0}]'
assert b.child('', 'pair') == '[{"kind":"place","shape":"pair","order":12,"to":[0,12]},{"kind":"extents","rows":1,"columns":13}]'
assert b.child('pair', 'x') == '[{"kind":"grid","container":"pair","span":1,"grows":"columns","cell_rotation":0.0},{"kind":"place","shape":"x","order":0,"to":[0,0]},{"kind":"extents","rows":1,"columns":1}]'
assert b.child('pair', 'y') == '[{"kind":"place","shape":"y","order":1,"to":[0,1]},{"kind":"extents","rows":1,"columns":2}]'
assert b.cells("pair") == '[[[0,0],[["x",0]],null,1,1],[[0,1],[["y",1]],null,1,1]]'
fresh = Bench()
laid = fresh.layout("", 2, "columns")
assert text([__import__("json").loads(x) for x in (laid, fresh.child("", "x"), fresh.child("", "y"), fresh.child("", "z"), fresh.cells(""))]) == '[[{"kind":"grid","container":"overview","span":2,"grows":"columns","cell_rotation":0.0}],[{"kind":"place","shape":"x","order":0,"to":[0,0]},{"kind":"extents","rows":1,"columns":1}],[{"kind":"place","shape":"y","order":1,"to":[1,0]},{"kind":"extents","rows":2,"columns":1}],[{"kind":"place","shape":"z","order":2,"to":[0,1]},{"kind":"extents","rows":2,"columns":2}],[[[0,0],[["x",0]],null,1,1],[[0,1],[["z",2]],null,1,1],[[1,0],[["y",1]],null,1,1]]]'

<a id="CNT-08"></a>
## CNT-08 · A shape moved into another container keeps its handle and what it owns: it leaves its cell, its container's or the diagram's, and takes its new container's next cell

In [ ]:
assert b.move('cpu.a', 'bus') == '[{"kind":"leave","shape":"a","from":[0,0]},{"kind":"move","shape":"c","from":[0,1],"to":[0,0]},{"kind":"move","shape":"d","from":[1,0],"to":[0,1]},{"kind":"extents","rows":1,"columns":2},{"kind":"place","shape":"a","order":5,"to":[1,1]}]'
assert b.cells("cpu") == '[[[0,0],[["c",2]],null,1,1],[[0,1],[["d",3]],null,1,1]]'
assert b.cells("bus") == '[[[0,0],[["b",1]],null,1,1],[[0,1],[["e",4]],null,1,1],[[1,0],[["c",2]],null,1,1],[[1,1],[["a",5]],null,1,1]]'
assert text(Logical.name_of(Logical.container_of(b.shape("cpu.a")))) == '"bus"'
assert b.move('solo', 'bus') == '[{"kind":"leave","shape":"solo","from":[0,11]},{"kind":"move","shape":"pair","from":[0,12],"to":[0,11]},{"kind":"extents","rows":1,"columns":12},{"kind":"place","shape":"solo","order":6,"to":[0,2]},{"kind":"extents","rows":2,"columns":3}]'
assert b.cells("") == '[[[0,0],[["cpu",0]],null,1,1],[[0,1],[["one",1]],null,1,1],[[0,2],[["bus",2]],null,1,1],[[0,3],[["b",3]],null,1,1],[[0,4],[["e",4]],null,1,1],[[0,5],[["a",5]],null,1,1],[[0,6],[["d",6]],null,1,1],[[0,7],[["x",7]],null,1,1],[[0,8],[["diagonal",8]],null,1,1],[[0,9],[["empty",9]],null,1,1],[[0,10],[["kids",10]],null,1,1],[[0,11],[["pair",12]],null,1,1]]'
assert b.cells("bus") == '[[[0,0],[["b",1]],null,1,1],[[0,1],[["e",4]],null,1,1],[[0,2],[["solo",6]],null,1,1],[[1,0],[["c",2]],null,1,1],[[1,1],[["a",5]],null,1,1]]'
assert b.cells("solo") == '[[[0,0],[["x",0]],null,1,1]]'
assert text(b.problems("cpu", "bus", "solo")) == '[[],[],[]]'

<a id="CNT-09"></a>
## CNT-09 · The journal, replayed in a fresh store, makes the same adjustments

In [ ]:
assert text(b.replay()) == '[]'